In [20]:
import pandas as pd
import numpy as np


df = pd.read_pickle('/Users/kkreth/PycharmProjects/cgan/dataset/3p6' + ".pkl", compression="zip")
#Now let's drop what I know I won't need
df = df.drop(columns=['px','py','pz','distance'])
df_subset = df[['vx','vy','vz']]

In [2]:
#Again, so slow as to be unusable

'''

#Now to normalize the values for our subset
# Define a lambda function to normalize a single value
normalize = lambda x: (x - df_subset.values.min()) / (df_subset.values.max() - df_subset.values.min())

# Apply the normalization function to each element of the DataFrame
df_normalized = df.applymap(normalize)

print(df_normalized)
'''

KeyboardInterrupt: 

In [ ]:
#This code was SOOOOOO slow
'''
import concurrent.futures

# Define a function to parallelize the normalization process
def parallel_normalize(col):
    min_val = np.min(col)
    max_val = np.max(col)
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = [executor.submit(normalize, x, min_val, max_val) for x in col]
    results = [f.result() for f in futures]
    return results


data = df_subset.values
# Normalize the data using parallel processing
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(parallel_normalize, col) for col in data.T]
results = [f.result() for f in futures]
df_normalized = pd.DataFrame(np.array(results).T, columns=df_subset.columns)

print(df_normalized)
'''

In [6]:
import torch

#First some housekeeping
df_subset.vx = df_subset.vx.astype('float32')
df_subset.vy = df_subset.vy.astype('float32')
df_subset.vz = df_subset.vz.astype('float32')

# Example tensor on CUDA device
tensor_cuda = torch.tensor(df_subset.values, device='mps')

# Calculate the minimum and maximum values of the tensor
min_val = torch.min(tensor_cuda)
max_val = torch.max(tensor_cuda)

# Perform min-max normalization using tensor operations
tensor_normalized = (tensor_cuda - min_val) / (max_val - min_val)

# Move the normalized tensor back to CPU device and print it
tensor_cpu = tensor_normalized.cpu()
print(tensor_cpu)

/var/folders/_4/8y8sz38n4cq0w_6k9w_80jfc0000gn/T/ipykernel_19618/2631247859.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset.vx = df_subset.vx.astype('float32')
/var/folders/_4/8y8sz38n4cq0w_6k9w_80jfc0000gn/T/ipykernel_19618/2631247859.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset.vy = df_subset.vy.astype('float32')
/var/folders/_4/8y8sz38n4cq0w_6k9w_80jfc0000gn/T/ipykernel_19618/2631247859.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a D

tensor([[0.7565, 0.3739, 0.3565],
        [0.7565, 0.3739, 0.3565],
        [0.7652, 0.3826, 0.3565],
        ...,
        [0.7652, 0.3304, 0.4087],
        [0.7652, 0.3304, 0.4087],
        [0.7652, 0.3304, 0.4000]])


In [14]:
size = tensor_cpu.numel()
print(size)
print(tensor_cpu.shape)

#About 180MM seems just right

182469600
torch.Size([60823200, 3])


In [8]:
mint = tensor_cpu.min()
print(mint)

maxt = tensor_cpu.max()
print(maxt)

tensor(0.)
tensor(1.)


In [19]:
x_norm = tensor_cpu.numpy()[:,0]
print(x_norm)

y_norm = tensor_cpu.numpy()[:,1]
print(y_norm)

z_norm = tensor_cpu.numpy()[:,2]
print(z_norm)

[0.75652176 0.75652176 0.7652174  ... 0.7652174  0.7652174  0.7652174 ]
[0.37391305 0.37391305 0.3826087  ... 0.3304348  0.3304348  0.3304348 ]
[0.35652176 0.35652176 0.35652176 ... 0.40869567 0.40869567 0.40000004]


In [21]:
#With NOT a lot of testing (other than the above) this looks corect.
#Now to place these values in the dataframe next to their non-normalized cousins.

df['vx_norm'] = x_norm
df['vy_norm'] = y_norm
df['vz_norm'] = z_norm

In [22]:
print(df)

              x     y     z    vx    vy    vz  time   vx_norm   vy_norm  \
0        -117.0  87.0 -33.0  0.46  0.02 -0.00   363  0.756522  0.373913   
1        -113.0  87.0 -33.0  0.46  0.02  0.00   363  0.756522  0.373913   
2        -109.0  87.0 -33.0  0.47  0.03  0.00   363  0.765217  0.382609   
3        -105.0  87.0 -33.0  0.47  0.03  0.00   363  0.765217  0.382609   
4        -101.0  87.0 -33.0  0.47  0.03  0.00   363  0.765217  0.382609   
...         ...   ...   ...   ...   ...   ...   ...       ...       ...   
60823195  109.0 -83.0  34.0  0.48 -0.03  0.06  1131  0.773913  0.330435   
60823196  113.0 -83.0  34.0  0.48 -0.03  0.06  1131  0.773913  0.330435   
60823197  117.0 -83.0  34.0  0.47 -0.03  0.06  1131  0.765217  0.330435   
60823198  121.0 -83.0  34.0  0.47 -0.03  0.06  1131  0.765217  0.330435   
60823199  125.0 -83.0  34.0  0.47 -0.03  0.05  1131  0.765217  0.330435   

           vz_norm  
0         0.356522  
1         0.356522  
2         0.356522  
3         0.356

In [24]:
print(df.describe())

                  x             y             z            vx            vy  \
count  6.082320e+07  6.082320e+07  6.082320e+07  6.082320e+07  6.082320e+07   
mean  -2.139052e-01  1.632995e+00  6.679951e-01  4.379836e-01  3.452447e-03   
std    7.320801e+01  5.034335e+01  2.036813e+01  1.074044e-01  6.553460e-02   
min   -1.250000e+02 -8.300000e+01 -3.300000e+01 -2.700000e-01 -4.100000e-01   
25%   -6.200000e+01 -4.400000e+01 -1.700000e+01  4.100000e-01 -3.000000e-02   
50%    2.000000e+00 -0.000000e+00  3.000000e+00  4.800000e-01  1.000000e-02   
75%    6.500000e+01  4.300000e+01  1.800000e+01  5.000000e-01  4.000000e-02   
max    1.250000e+02  8.700000e+01  3.400000e+01  7.400000e-01  4.000000e-01   

                 vz          time       vx_norm       vy_norm       vz_norm  
count  6.082320e+07  6.082320e+07  6.082320e+07  6.082320e+07  6.082320e+07  
mean   1.749684e-02  6.005000e+02  7.373785e-01  3.595234e-01  3.717363e-01  
std    4.455674e-02  3.464100e+02  9.339515e-02  5.698

In [26]:
print(df.dtypes)

x          float64
y          float64
z          float64
vx         float64
vy         float64
vz         float64
time         int64
vx_norm    float32
vy_norm    float32
vz_norm    float32
dtype: object


In [27]:
#File size was 8GB...so not using this afterall...AND it doesn't keep the damn types...
#df.to_json('/Users/kkreth/PycharmProjects/cgan/dataset/3p6_with_normalized_values.json', orient='records')

In [33]:
import gzip
import pickle
# Save the DataFrame to disk with pickle and gzip compression

df.to_pickle('/Users/kkreth/PycharmProjects/cgan/dataset/3p6_with_normalized_values.pkl.gz', compression='gzip', protocol=5, storage_options=None)